In [ ]:
# install PyTorch: pip install torch
# install scikit-learn: pip install scikit-learn

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load and return the breast cancer Wisconsin dataset (as an example) using sklearn.datasets.load_breast_cancer
X, y = load_breast_cancer(return_X_y=True)  # X: Numpy 2d array (569 x 30) representing 569 samples and 30 features for each sample; y: Numpy 1d array containing 569 target labels

# Split Numpy ndarrays X & y each into two sets (train & test); test_size: how much percentage for the test data (20% here)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2)  

# Standardize X_train & X_test datasets; ensure individual features look like standard normally distributed data (e.g., Gaussian)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit on train data to get the mean and standard deviation (std) and then perform transfomation
X_test_scaled = scaler.transform(X_test)  # perform tranformation with the above already-learned mean and std for test data

# Convert data type
X_train_scaled_tensor = torch.from_numpy(X_train_scaled).float()  # convert Numpy 2d array to PyTorch 2d tensor
X_test_scaled_tensor = torch.from_numpy(X_test_scaled).float()
y_train_tensor = torch.from_numpy(y_train).float().unsqueeze(1)  # y-label: originally Numpy 1d array; converted to PyTorch 1d tensor; then expanded to 2d (n x 1) tensor
y_test_tensor = torch.from_numpy(y_test).float().unsqueeze(1)

# Generate formatted "dataset_train" which is needed by class DataLoader in the next step
# Implement it with class TensorDataset (inheriting class torch.utils.data.Dataset with two essential methods __len__() and __getitem__()) 
dataset_train = TensorDataset(X_train_scaled_tensor, y_train_tensor)

# Instantiate class DataLoader; load data in batches: 16 samples per batch here
dataloader_train = DataLoader(dataset_train, batch_size=16, shuffle=True)

# Define a class-based neural network
class BCNet(nn.Module):
    def __init__(self):  # automatically run when an instance is created
        super().__init__()
        self.fc1 = nn.Linear(30, 64)  # fully-connected layer 1: 64 neurons; each receives 30 numbers from the input layer (showing 30 features for each sample)
        self.fc2 = nn.Linear(64, 16)  # fully-connected layer 2: 16 neurons; each receives 64 numbers (i.e., outputs of the previous layer)
        self.fc3 = nn.Linear(16, 1)  # fully-connected layer 3 (the output layer): a single neuron

    def forward(self, x):
        x = F.relu(self.fc1(x))  # apply activation function ReLU
        x = F.relu(self.fc2(x))
        x = F.sigmoid(self.fc3(x))  # apply sigmoid function to the output-layer output value -> implement binary classification

        return x

net = BCNet()  # instantiate class BCNet

criterion = nn.BCELoss()  # class BCELoss; use the binary cross entropy loss function (for the current "binary classification" problem)
optimizer = optim.Adam(net.parameters(), lr=0.001)  # optimization with Adam (Adaptive moment estimation) algorithm; lr: learning rate

print("Model Training Process:\n")

# Training loop
for epoch in range(20):
    net.train()  # model training
    running_loss = 0
    for x_batch, y_batch in dataloader_train:  # for each batch (including multiple samples) of loaded data
        optimizer.zero_grad()  # clear old gradients (they accumulate by default)
        preds = net(x_batch)  # forward pass to get predicted results
        loss = criterion(preds, y_batch)  # calculate loss (0-dimensional tensor), based on predicted y values and true y labels
        loss.backward()  # backpropagation to calculate gradients for all weights and biases used in linear neural network
        optimizer.step()  # update all parameters (weights and biases); NOTE: these new parameters used as the starting point for the next batch
        running_loss += loss.item()  # loss.item(): scalar (0-dimensional) tensor -> a plain float
    print(f"Epoch {epoch+1}: Loss is {running_loss}")

# Validation
with torch.no_grad():  # disable gradient tracking to save memory (no gradient is needed during validation)
    net.eval()  # model evaluation
    preds = net(X_test_scaled_tensor)
    new_preds = (preds >= 0.5).float()  # re-set the predicted values (>= 0.5 or < 0.5) to be 1 or 0, matching target y-lables
    loss = criterion(new_preds, y_test_tensor).item()
    print(f"\nModel Evaluation with Test Data: Loss = {loss}")
